In [15]:
# ===============================
# 1. LOAD LIBRARIES
# ===============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [16]:

# ===============================
# 2. LOAD DATA
# ===============================
df = pd.read_csv("QualityOfCare.csv")

In [19]:
df.columns

Index(['Health facility level', 'FacilityType', 'Age', 'Sex', 'MaritalStatus',
       'EducationLevel', 'Occupation', 'CareEntryPoint', 'WeightAtStart',
       'HeightAtStart', 'FunctionalStatusAtStart', 'Cd4AtStart',
       'AdherenceCouncelingCompleted', 'InitialTbScreeningDone',
       'ArtSubstitution', 'ArtSwitch', 'ArtInterruption', 'WeightAtLastVisit',
       'HeightAtLastVisit', 'OpportunisticInfectionPresentAtLastVisit',
       'AnySideEffects', 'WasPatientReceivingArv', 'ArvAdherenceLatestLevel',
       'MostRecentCd4Count', 'ViralLoadSuppression'],
      dtype='object')

In [21]:

# ===============================
# 3. DEFINE & ENCODE TARGET
# ===============================
TARGET = "ViralLoadSuppression"

# Encode target: Yes = 1, No = 0
df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})

In [23]:
# ===============================
# 4. IDENTIFY VARIABLE TYPES
# ===============================
categorical_cols = df.select_dtypes(include="object").columns
continuous_cols = df.select_dtypes(exclude="object").drop(TARGET, axis=1).columns

In [24]:
# ===============================
# 5. HANDLE MISSING VALUES
# ===============================
df[categorical_cols] = df[categorical_cols].apply(
    lambda x: x.fillna(x.mode()[0])
)

df[continuous_cols] = df[continuous_cols].apply(
    lambda x: x.fillna(x.mean())
)


In [25]:

# ===============================
# 6. TARGET ENCODING (BARRIER-ORIENTED)
# ===============================
for col in categorical_cols:
    df[col] = df.groupby(col)[TARGET].transform("mean")

In [26]:
# ===============================
# 7. FEATURE SCALING
# ===============================
scaler = StandardScaler()
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

In [ ]:
# REMOVE ROWS WITH MISSING TARGET VALUES
df = df.dropna(subset=[TARGET])

# DEFINE X AND y
X = df.drop(TARGET, axis=1)
y = df[TARGET]


In [ ]:
#Data Splitting

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)


In [30]:
# ===============================
# 9. LOGISTIC REGRESSION (INFERENCE)
# ===============================
X_train_sm = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_train_sm).fit(disp=False)

In [35]:
# ===============================
# ODDS RATIOS, P-VALUES & CIs
# ===============================
params = logit_model.params
conf = logit_model.conf_int()
conf.columns = ["CI_2.5%", "CI_97.5%"]

results = pd.DataFrame({
    "Coefficient": params,
    "Std_Error": logit_model.bse,
    "Z_value": logit_model.tvalues,
    "P_value": logit_model.pvalues,
    "Odds_Ratio": np.exp(params),
    "CI_2.5%": np.exp(conf["CI_2.5%"]),
    "CI_97.5%": np.exp(conf["CI_97.5%"])
})

results = results.drop("const").sort_values("P_value")

print(results)


                                          Coefficient  Std_Error    Z_value  \
ArtInterruption                              3.316710   0.268797  12.339067   
AdherenceCouncelingCompleted                 3.899047   0.328664  11.863313   
ArvAdherenceLatestLevel                      3.460678   0.299204  11.566268   
Health facility level                        3.637239   0.326118  11.153136   
WasPatientReceivingArv                       3.115288   0.318196   9.790475   
CareEntryPoint                               3.855848   0.414337   9.306064   
ArtSubstitution                              4.822032   0.573125   8.413575   
FacilityType                                 3.778096   0.524461   7.203772   
EducationLevel                               2.710174   0.431186   6.285392   
Sex                                          4.578738   0.747354   6.126597   
FunctionalStatusAtStart                      3.797594   0.692560   5.483415   
MaritalStatus                                2.33333

In [ ]:
import joblib

joblib.dump(logit_model, "viral_load_suppression_logit_model.pkl")

joblib.dump(logit_model, "logit_model.joblib")



['logit_model.joblib']